# Group 19 |  How to run our code | Step-by-step tutorial




**Step 1:**  Download the data from the [Reef Support Google Drive](https://drive.google.com/drive/folders/1mx2OJcVKp1mRbTbjezqWucDXpbGrd_OA) *maks_labels* and put those data to the folder "website" so that our files can find correct path. 


**Step 2:**  Download all important libraries

In [ ]:
import argparse, os, sys, glob, cv2, csv, re, random
from ultralytics import YOLO
import pandas as pd
import numpy as np
from skimage import measure
from pathlib import Path
import matplotlib.pyplot as plt
from collections import Counter


**Step 3:** We need to create CSV file which will be our "dictionary"  with following columns:

|type_x | name | name_ext | path| label_name| label_name_ext | label_path | polygon_count | type_y |soft_area | hard_area | total_area|

**type_X** - Whether file is mask or image\
**name** - name of the file e.g Coral1\
**name_ext** - name of the file with file extension e.g Coral1.png\
**path** - path to the file\
**label_name** - if an object is image then there is a label_name which corresponds to the name of the file e.g Coral1_mask\
**label_name_ect** - if an object is image then there is label_name which corresponding name and file extension e.g Coral1_mask.txt\
**label_path** - path to the label\
**polygon_count** - the number of corals in one picture e.g 53\
**type_y** - [soft-only, hard-only, mixed] corals\
**soft_area** - % of what soft corals are in the picture\
**hard_area** - % of what hard corals are in the picture\
**total_area** - % of what corals are taking the picture combined (soft_area + hard_area)

Code to create "output.csv" (Remember to adjust ROOT)



In [ ]:
ROOT = r"C:\Users\20231807\Documents\GitHub\Cap_coral_reefs\New_version_of_the_project\website\benthic_datasets\mask_labels\reef_support"

def creating_dataframe():
    df = pd.DataFrame(columns=["type", "name", "name_ext", "path"])

    dirlist = os.listdir(ROOT)

    for dir in dirlist:
        new_address = os.path.join(ROOT, dir)
        if os.path.exists(new_address):
            print(f"There is dir {new_address}")
            images_path = os.path.join(new_address, "images")
            masks_stitched_path = os.path.join(new_address, "masks_stitched")
            labels_path = os.path.join(new_address,"labels")
            print(f"Lets check: images path -> {images_path} and masks_stitched path {masks_stitched_path}")
            print("\n")

            # Convert to Path objects
            images_path = Path(images_path)
            masks_stitched_path = Path(masks_stitched_path)

            # Iterate over masks in dir
            if masks_stitched_path.exists():  # Check if directory exists
                for mask in masks_stitched_path.iterdir():
                    if mask.is_file():
                        mask_name_ext = mask.name
                        print(mask.name)
                        mask_name = mask.stem
                        print(mask.stem)
                        mask_path = str(mask)  # Convert to string for DataFrame
                        print(str(mask))

                        # Create new row
                        new_row_mask = pd.DataFrame({
                            "type": ["mask"],  # Note: lists for DataFrame creation
                            "name": [mask_name],
                            "name_ext": [mask_name_ext],
                            "path": [mask_path]
                        })

                        df = pd.concat([df, new_row_mask], ignore_index=True)

            # Iterate over images in dir
            if images_path.exists():  # Check if directory exists
                for image in images_path.iterdir():
                    if image.is_file():
                        image_name_ext = image.name
                        print(image.name)
                        image_name = image.stem  # FIXED: was image.name
                        print(image.stem)
                        image_path = str(image)  # Convert to string for DataFrame
                        print(str(image))

                        # Create new row
                        new_row_image = pd.DataFrame({
                            "type": ["image"],  # Note: lists for DataFrame creation
                            "name": [image_name],
                            "name_ext": [image_name_ext],
                            "path": [image_path]
                        })

                        df = pd.concat([df, new_row_image], ignore_index=True)


    df.to_csv("output.csv",index=False)

    print(df)

# Calling the function

creating_dataframe()

def adding_labels():
    df = pd.read_csv("output.csv")

    dirlist = os.listdir(ROOT)

    for dir in dirlist:
        new_address = os.path.join(ROOT, dir)
        if os.path.exists(new_address):
            print(f"There is dir {new_address}")
            labels_path = Path(os.path.join(new_address, "labels"))  # Convert to Path
            print("\n")

            if labels_path.exists():
                for text in labels_path.iterdir():
                    if text.is_file():
                        text_name_ext = text.name
                        text_name = text.stem
                        text_path = str(text)  # Convert to string

                        # Find rows where name matches text_name
                        mask = df['name'] == text_name

                        # Add new columns if they don't exist
                        if 'label_name' not in df.columns:
                            df['label_name'] = None
                        if 'label_name_ext' not in df.columns:
                            df['label_name_ext'] = None
                        if 'label_path' not in df.columns:
                            df['label_path'] = None

                        # Update matching rows
                        df.loc[mask, 'label_name'] = text_name
                        df.loc[mask, 'label_name_ext'] = text_name_ext
                        df.loc[mask, 'label_path'] = text_path

                        print(f"Updated {mask.sum()} rows for {text_name}")

    # Save the updated DataFrame
    df.to_csv("output.csv", index=False)


# Call the function
adding_labels()


 In order to finish this csv we might need to use a help of additional one "poly.csv" which focuses on each coral and additionally we create some EDA

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CSV_PATH = "output.csv"
FIG_DIR = "figs"
os.makedirs(FIG_DIR, exist_ok=True)

def polygon_area(xs, ys):
    n = len(xs)
    if n < 3 or n != len(ys):
        return 0.0
    if xs[0] == xs[-1] and ys[0] == ys[-1]:
        xs, ys = xs[:-1], ys[:-1]
        n -= 1
        if n < 3:
            return 0.0
    return 0.5 * abs(sum(xs[i] * ys[(i + 1) % n] - xs[(i + 1) % n] * ys[i] for i in range(n)))

def read_label_file(path):
    hard_area = 0.0
    soft_area = 0.0
    with open(path, "r") as f:
        for ln, line in enumerate(f, 1):
            parts = line.strip().split()
            if not parts:
                continue
            if len(parts) < 7:
                continue
            try:
                cls = int(parts[0])
                coords = list(map(float, parts[1:]))
            except ValueError:
                continue
            if len(coords) % 2 != 0:
                continue
            xs = coords[0::2]
            ys = coords[1::2]
            a = polygon_area(xs, ys)
            if a <= 0:
                continue
            if cls == 0:
                hard_area += a
            elif cls == 1:
                soft_area += a
    return hard_area, soft_area

def parse_single_label_file(label_path: str, image_path: str, image_name: str) -> pd.DataFrame:
    """
    Reads one polygon .txt and returns rows:
      polygon_id, file, file_path, label_path, class_id, n_vertices, coords (np.ndarray Nx2)
    """
    p = Path(label_path)
    lines = Path(label_path).read_text().strip().splitlines()
    rows = []
    for i, line in enumerate(lines):
        vals = [float(v) for v in line.strip().split()]
        cls = int(vals[0])
        coords = np.array(vals[1:], dtype=float).reshape(-1, 2)
        rows.append({
            "polygon_id": f"{image_name}_{i}",
            "file": image_name,
            "file_path": str(image_path),
            "label_path": str(label_path),
            "class_id": cls,
            "coords": coords,
            "n_vertices": len(coords),
        })
    return pd.DataFrame(rows)

def parse_from_output_csv(csv_path: str) -> pd.DataFrame:
    """
    Reads output.csv with columns:
      ['type','name','name_ext','path','label_name','label_name_ext','label_path']
    Iterates rows and parses each label file into polygons.
    """
    meta = pd.read_csv(csv_path)
    all_rows = []
    for _, r in meta.iterrows():
        img_path = r["path"]
        lbl_path = r["label_path"]
        img_name = Path(img_path).stem
        if pd.isna(lbl_path) or not Path(lbl_path).exists():
            continue
        df_one = parse_single_label_file(lbl_path, img_path, img_name)
        all_rows.append(df_one)
    if not all_rows:
        return pd.DataFrame(columns=[
            "polygon_id","file","file_path","label_path","class_id","coords","n_vertices"
        ])
    return pd.concat(all_rows, ignore_index=True)

def class_imbalance():
    df = pd.read_csv("polys.csv")
    class_counts = df["class_id"].value_counts()
    print(f'Class Counts:\n{class_counts}')
    class_counts.plot(kind='bar', title='Class Imbalance', rot=0)
    plt.ylabel('Number of Samples')
    plt.xlabel('Class ID (0=Hard, 1=Soft)')
    plt.show()

def polygon_count():
    df = pd.read_csv("polys.csv")
    df_output = pd.read_csv("output.csv")
    counts = df.groupby("file").size()
    df_output = df_output.merge(
        counts.rename("polygon_count"),
        left_on="name",
        right_index=True,
        how="left"
    )
    df_output["polygon_count"] = df_output["polygon_count"].fillna(0).astype(int)
    df_output.to_csv("output.csv", index=False)

def classify_files():
    df = pd.read_csv("polys.csv")
    df_output = pd.read_csv("output.csv")
    classes_per_file = df.groupby("file")["class_id"].apply(set)
    classification = classes_per_file.apply(
        lambda s: "hard-only" if s == {0}
        else "soft-only" if s == {1}
        else "mixed"
    )
    df_output = df_output.merge(
        classification.rename("type"),
        left_on="name",
        right_index=True,
        how="left"
    )
    df_output.to_csv("output.csv", index=False)

def area_coral():
    df = pd.read_csv("output.csv")
    mask_labels = df.loc[df["type_x"] == "image", "label_path"]
    df["soft_area"] = None
    df["hard_area"] = None
    df["total_area"] = None
    for idx, path in mask_labels.items():
        hard, soft = read_label_file(path)
        df.loc[idx, "soft_area"] = soft
        df.loc[idx, "hard_area"] = hard
        df.loc[idx, "total_area"] = soft + hard
    df.to_csv("output.csv", index=False)
    return None

def plots():
    df = pd.read_csv("output.csv")
    df["polygon_count"] = pd.to_numeric(df["polygon_count"], errors="coerce")
    bins = list(range(1, 171, 10))
    labels = [f"{low}-{low + 9}" for low in bins[:-1]]
    df["polygon_bin"] = pd.cut(df["polygon_count"], bins=bins, labels=labels, include_lowest=True)
    grouped = df.groupby(["polygon_bin", "type_y"]).size().unstack(fill_value=0)
    grouped.plot(kind="bar", stacked=True, figsize=(10, 6))
    plt.title("Polygon Count Distribution by Type and Bins")
    plt.xlabel("Polygon Count Range")
    plt.ylabel("Number of Samples")
    plt.xticks(rotation=45)
    plt.legend(title="type_y")
    plt.show()

def load_and_prepare(csv_path):
    df = pd.read_csv(csv_path)
    for col in ["polygon_count", "soft_area", "hard_area", "total_area"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    if {"total_area", "soft_area", "hard_area"}.issubset(df.columns):
        need = df["total_area"].isna() & df["soft_area"].notna() & df["hard_area"].notna()
        df.loc[need, "total_area"] = df.loc[need, "soft_area"] + df.loc[need, "hard_area"]
    return df

def ecdf(values):
    x = np.sort(values)
    y = np.arange(1, len(x)+1) / len(x)
    return x, y

def save_show(name):
    plt.tight_layout()
    p = os.path.join(FIG_DIR, name)
    plt.savefig(p, bbox_inches="tight", dpi=150)
    plt.close()
    return p

def plot_histograms(df):
    for col in ["soft_area", "hard_area", "total_area"]:
        if col in df.columns and df[col].notna().any():
            plt.figure()
            df[col].dropna().plot(kind="hist", bins=40, alpha=0.9)
            plt.title(f"Distribution of {col}"); plt.xlabel(col); plt.ylabel("Count")
            save_show(f"hist_{col}.png")

def plot_ecdfs(df):
    for col in ["soft_area", "hard_area", "total_area"]:
        if col in df.columns and df[col].notna().any():
            x, y = ecdf(df[col].dropna().values)
            plt.figure()
            plt.step(x, y, where="post")
            plt.title(f"ECDF of {col}"); plt.xlabel(col); plt.ylabel("F(x)")
            save_show(f"ecdf_{col}.png")

def plot_box_violin_by_type(df):
    if "type_y" not in df.columns:
        return
    dff = df.dropna(subset=["type_y"])
    for col in ["soft_area", "hard_area", "total_area"]:
        if col in dff.columns and dff[col].notna().any():
            plt.figure()
            dff.boxplot(column=col, by="type_y", rot=45)
            plt.title(f"{col} by type_y"); plt.suptitle(""); plt.ylabel(col); plt.xlabel("type_y")
            save_show(f"box_{col}_by_type.png")
            groups = [g[col].dropna().values for _, g in dff.groupby("type_y")]
            labels = [str(k) for k, _ in dff.groupby("type_y")]
            if all(len(g) for g in groups) and len(groups) >= 2:
                plt.figure()
                plt.violinplot(groups, showmeans=True, showextrema=True, showmedians=True)
                plt.xticks(range(1, len(labels)+1), labels, rotation=45)
                plt.title(f"Violin: {col} by type_y"); plt.ylabel(col)
                save_show(f"violin_{col}_by_type.png")

def plot_soft_vs_hard(df):
    if {"soft_area","hard_area"}.issubset(df.columns):
        d = df.dropna(subset=["soft_area","hard_area"])
        if len(d) == 0:
            return
        plt.figure()
        plt.scatter(d["soft_area"], d["hard_area"], alpha=0.6)
        lim = max(d["soft_area"].max(), d["hard_area"].max())
        plt.plot([0, lim], [0, lim])
        plt.title("Soft vs Hard Area (scatter)"); plt.xlabel("soft_area"); plt.ylabel("hard_area")
        save_show("scatter_soft_vs_hard.png")
        plt.figure()
        plt.hexbin(d["soft_area"], d["hard_area"], gridsize=40)
        plt.plot([0, lim], [0, lim])
        plt.xlabel("soft_area"); plt.ylabel("hard_area"); plt.title("Soft vs Hard Area (hexbin)")
        cb = plt.colorbar(); cb.set_label("count in bin")
        save_show("hexbin_soft_vs_hard.png")

def plot_means_by_type(df):
    if "type_y" not in df.columns:
        return
    cols = [c for c in ["soft_area","hard_area","total_area"] if c in df.columns]
    if not cols:
        return
    means = df.groupby("type_y")[cols].mean(numeric_only=True)
    stds  = df.groupby("type_y")[cols].std(numeric_only=True)
    ax = means.plot(kind="bar")
    for col in cols:
        plt.errorbar(np.arange(len(means.index)), means[col].values, yerr=stds[col].values,
                     fmt="none", ecolor="black", capsize=3)
    plt.title("Mean areas by type_y (±1 SD)"); plt.ylabel("Area"); plt.xlabel("type_y")
    plt.xticks(rotation=45)
    save_show("means_by_type.png")

def plot_polygon_bins_by_type(df):
    if "polygon_count" not in df.columns:
        return
    bins = list(range(1, 131, 10))
    labels = [f"{b}-{b+9}" for b in bins[:-1]]
    tmp = df.copy()
    tmp["polygon_bin"] = pd.cut(tmp["polygon_count"], bins=bins, labels=labels, include_lowest=True)
    tmp = tmp.dropna(subset=["polygon_bin"])
    if "type_y" in tmp.columns and not tmp.empty:
        tab = tmp.groupby(["polygon_bin","type_y"]).size().unstack(fill_value=0)
        tab.plot(kind="bar", stacked=False, rot=45)
        plt.title("Polygon count distribution by bin and type_y (grouped)")
        plt.xlabel("polygon_count bin"); plt.ylabel("Count")
        save_show("bars_polygon_bins_grouped.png")
        perc = tab.div(tab.sum(axis=1), axis=0)
        perc.plot(kind="bar", stacked=True, rot=45)
        plt.title("Polygon count composition by bin (100% stacked)")
        plt.xlabel("polygon_count bin"); plt.ylabel("Share")
        save_show("bars_polygon_bins_100pct.png")

def plot_correlation(df):
    cols = [c for c in ["polygon_count","soft_area","hard_area","total_area"] if c in df.columns]
    if not cols:
        return
    corr = df[cols].corr(numeric_only=True)
    plt.figure()
    plt.imshow(corr, interpolation="nearest")
    plt.xticks(range(len(cols)), cols, rotation=45, ha="right"); plt.yticks(range(len(cols)), cols)
    plt.title("Correlation matrix"); cb = plt.colorbar(); cb.set_label("corr")
    save_show("correlation_matrix.png")

def run_all():
    df = load_and_prepare(CSV_PATH)
    plot_histograms(df)
    plot_ecdfs(df)
    plot_box_violin_by_type(df)
    plot_soft_vs_hard(df)
    plot_means_by_type(df)
    plot_polygon_bins_by_type(df)
    plot_correlation(df)

if __name__ == "__main__":
    run_all()


We are done with the first part! Now let's focus on model 


To do so we need to have few important things:
1. Model [Download](https://huggingface.co/reefsupport/coral-ai/resolve/main/models/yolov8_sm_latest.pt)
2. All traning,val,test data [Download](https://tuenl-my.sharepoint.com/:f:/g/personal/j_i_kuczynski_student_tue_nl/EtoFKoLvgR1GpKCXpZfeZfoBv485qEcsh4mnGzUiKx-TTQ?e=hR5zaI)
This files you need to put to the folder "splits" 
3. Create data.yaml which you can also Download [here](https://tuenl-my.sharepoint.com/:f:/g/personal/j_i_kuczynski_student_tue_nl/EtoFKoLvgR1GpKCXpZfeZfoBv485qEcsh4mnGzUiKx-TTQ?e=hR5zaI)
And remember to adjust data.yaml to have correct path


**Step 4:** Create balanced train sets.\
Run this code to generate train sets that are balanced with linear and sqrt Repeat Factor Sampling (they will be in the splits folder):

In [ ]:
def read_lines(p):
    with open(p, "r", encoding="utf-8") as f:
        return [ln.strip() for ln in f if ln.strip()]


def image_to_label_path(img_path):
    s = img_path.replace("\\", "/")
    s = re.sub(r"[\\/]+images[\\/]+", "/labels/", s)
    s = re.sub(r"\.(jpg|jpeg|png|bmp)$", ".txt", s, flags=re.I)
    return s


def count_classes_in_label(lbl_path):
    cnt = Counter()
    if not os.path.exists(lbl_path):
        return cnt
    with open(lbl_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                cls = int(float(line.split()[0]))
                cnt[cls] += 1
            except Exception:
                pass
    return cnt


def build_index(train_list):
    per_img = []
    inst_tot = Counter()
    for img in train_list:
        lbl = image_to_label_path(img)
        c = count_classes_in_label(lbl)
        per_img.append((img, c))
        inst_tot.update(c)
    return per_img, inst_tot


def rfs_repeat_factors(inst_tot, alpha=0.5):
    """
    r_c = max(1, (f_min / f_c)^alpha)
    alpha=0.5 -> sqrt scaling (gentle)
    alpha=1.0 -> linear scaling (stronger)
    """
    tot_instances = sum(inst_tot.values())
    if tot_instances == 0:
        return {c: 1.0 for c in inst_tot}
    freq = {c: inst_tot[c] / tot_instances for c in inst_tot}
    f_min = min(freq.values()) if freq else 1.0
    r_class = {c: max(1.0, (f_min / max(freq[c], 1e-12)) ** alpha) for c in inst_tot}
    return r_class


def make_list(per_img, r_class, max_repeat=6, seed=0):
    out = []
    rng = random.Random(seed)
    for img, c in per_img:
        r = 1.0
        for cls, n in c.items():
            if n > 0:
                r = max(r, r_class.get(cls, 1.0) * (n ** 0.5))
        r = min(max_repeat, r)
        base = int(r)
        frac = r - base
        repeats = base + (1 if rng.random() < frac else 0)
        out.extend([img] * max(1, repeats))
    rng.shuffle(out)
    return out


def write_list(path, items):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(items))


def process_one(train_path, tag, args):
    base = read_lines(train_path)
    per_img, inst_tot = build_index(base)
    print(f"\n=== {tag} ===")
    print(f"Train list: {train_path}  (images: {len(base)})")
    print(f"Total instances: {dict(inst_tot)}")

    r_class_sqrt = rfs_repeat_factors(inst_tot, alpha=args.alpha_sqrt)
    r_class_lin = rfs_repeat_factors(inst_tot, alpha=args.alpha_linear)

    lst_sqrt = make_list(per_img, r_class_sqrt, max_repeat=args.sqrt_max_repeat, seed=args.seed)
    lst_linear = make_list(per_img, r_class_lin, max_repeat=args.linear_max_repeat, seed=args.seed)

    p_sqrt = os.path.join("splits", f"train_rfs_sqrt_{tag}.txt")
    p_linear = os.path.join("splits", f"train_rfs_linear_{tag}.txt")
    write_list(p_sqrt, lst_sqrt)
    write_list(p_linear, lst_linear)

    print(f"Wrote:\n  {p_sqrt}   (n={len(lst_sqrt)})\n  {p_linear} (n={len(lst_linear)})")
    return p_sqrt, p_linear


def main():
    ap = argparse.ArgumentParser(description="Build RFS train lists for one or more train splits.")
    ap.add_argument("--trains", nargs="+", required=True, help="Paths to train list(s), e.g. splits/train_*.txt")
    ap.add_argument("--tags", nargs="*", help="Tags to append (auto from filename if omitted)")
    ap.add_argument("--alpha_sqrt", type=float, default=0.5, help="Alpha for sqrt/gentle scaling")
    ap.add_argument("--alpha_linear", type=float, default=1.0, help="Alpha for linear/strong scaling")
    ap.add_argument("--sqrt_max_repeat", type=int, default=6, help="Max repeats for sqrt version")
    ap.add_argument("--linear_max_repeat", type=int, default=8, help="Max repeats for linear version")
    ap.add_argument("--seed", type=int, default=0)
    args = ap.parse_args()

    tags = args.tags or [Path(p).stem.replace("train_", "") for p in args.trains]
    if len(tags) != len(args.trains):
        raise SystemExit("[ERROR] Number of --tags must match number of --trains")

    for train_path, tag in zip(args.trains, tags):
        if not os.path.exists(train_path):
            print(f"[WARN] File not found: {train_path}")
            continue
        process_one(train_path, tag, args)


if __name__ == "__main__":
    main()


Run this code to generate train sets that are balanced with Random Oversampling (they will be in the splits folder):

In [ ]:
def read_lines(p):
    with open(p, "r", encoding="utf-8") as f:
        return [ln.strip() for ln in f if ln.strip()]


def image_to_label_path(img_path):
    s = img_path.replace("\\", "/")
    s = re.sub(r"[\\/]+images[\\/]+", "/labels/", s)
    s = re.sub(r"\.(jpg|jpeg|png|bmp)$", ".txt", s, flags=re.I)
    return s


def count_classes_in_label(lbl_path):
    c = Counter()
    if not os.path.exists(lbl_path):
        return c
    with open(lbl_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            parts = line.split()
            try:
                cls = int(float(parts[0]));
                c[cls] += 1
            except Exception:
                pass
    return c


def build_index(train_list):
    per_img, totals = [], Counter()
    for img in train_list:
        cnt = count_classes_in_label(image_to_label_path(img))
        per_img.append((img, cnt))
        totals.update(cnt)
    return per_img, totals


def compute_max_reachable_ratio(per_img, soft_id, hard_id, max_repeat_per_image):
    base = Counter()
    for _, cnt in per_img: base.update(cnt)
    S0, H0 = base[soft_id], (base[hard_id] if base[hard_id] > 0 else 1)
    S_add = H_add = 0
    for _, cnt in per_img:
        if cnt.get(soft_id, 0) > 0:
            S_add += cnt.get(soft_id, 0) * max_repeat_per_image
            H_add += cnt.get(hard_id, 0) * max_repeat_per_image
    S_max, H_max = S0 + S_add, H0 + H_add
    return S0, H0, S_max, H_max, (S_max / max(H_max, 1))


def random_oversample(per_img, soft_id, hard_id, target_ratio=1.0,
                      max_repeat_per_image=8, max_iterations=2_000_000,
                      progress_every=100_000):
    rng = random.Random(0)
    out = [img for img, _ in per_img]
    totals = Counter()
    soft_pool = []
    for img, cnt in per_img:
        totals.update(cnt)
        if cnt.get(soft_id, 0) > 0:
            soft_pool.append((img, cnt))
    if totals[soft_id] == 0:
        raise RuntimeError("No soft (minority) instances found—check class ids and labels.")
    if totals[hard_id] == 0:
        totals[hard_id] = 1

    def ratio(t):
        return t[soft_id] / max(t[hard_id], 1)

    # Reachability clamp
    _, _, _, _, max_reach = compute_max_reachable_ratio(per_img, soft_id, hard_id, max_repeat_per_image)
    if target_ratio > max_reach + 1e-9:
        print(f"[WARN] Target ratio {target_ratio:.3f} unreachable with max_repeat={max_repeat_per_image}. "
              f"Clamping to ≈ {max_reach:.3f}.")
        target_ratio = max_reach

    if ratio(totals) >= target_ratio:
        rng.shuffle(out);
        return out, dict(totals)

    repeats = Counter()
    iters = 0
    while ratio(totals) < target_ratio and iters < max_iterations:
        iters += 1
        img, cnt = rng.choice(soft_pool)
        if repeats[img] >= max_repeat_per_image:
            continue
        out.append(img);
        repeats[img] += 1;
        totals.update(cnt)
        if progress_every and (iters % progress_every) == 0:
            print(f"[oversample] iter={iters:,}  ratio={ratio(totals):.4f}  target={target_ratio:.4f}  "
                  f"capped_imgs={sum(1 for v in repeats.values() if v >= max_repeat_per_image)}")
    if iters >= max_iterations and ratio(totals) < target_ratio - 1e-6:
        print(f"[WARN] Stopped at {iters:,} iters; ratio={ratio(totals):.4f} < target={target_ratio:.4f}. "
              f"Try higher --max_repeat or lower --target.")
    rng.shuffle(out);
    return out, dict(totals)


def write_list(path, items):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(items))


def main():
    import argparse
    ap = argparse.ArgumentParser(description="Random oversampling for class balance (soft coral).")
    ap.add_argument("--train", required=True, help="Path to baseline train list")
    ap.add_argument("--soft_id", type=int, required=True, help="Soft coral class id (usually 1)")
    ap.add_argument("--hard_id", type=int, default=0, help="Hard coral class id (usually 0)")
    ap.add_argument("--tag", type=str, default="", help="Suffix for output filename (e.g., realistic)")
    ap.add_argument("--target", type=float, default=1.0, help="Target soft:hard ratio")
    ap.add_argument("--max_repeat", type=int, default=8, help="Max repeats per image")
    ap.add_argument("--max_iter", type=int, default=2_000_000, help="Failsafe iteration cap")
    ap.add_argument("--progress_every", type=int, default=50_000, help="Progress print frequency (0 to disable)")
    args = ap.parse_args()

    base = read_lines(args.train)
    per_img, base_tot = build_index(base)
    print("=== Baseline instance totals ===")
    for k in sorted(base_tot): print(f"class {k}: {base_tot[k]}")
    print(f"Images in baseline: {len(per_img)}")
    S0, H0, _, _, max_reach = compute_max_reachable_ratio(per_img, args.soft_id, args.hard_id, args.max_repeat)
    curr = S0 / max(H0, 1)
    print(f"Current soft:hard ≈ {curr:.3f} | Max reachable with max_repeat={args.max_repeat}: ≈ {max_reach:.3f}")

    out_list, out_tot = random_oversample(
        per_img, args.soft_id, args.hard_id,
        target_ratio=args.target,
        max_repeat_per_image=args.max_repeat,
        max_iterations=args.max_iter,
        progress_every=args.progress_every
    )
    tag = f"__{args.tag}" if args.tag else ""
    out_path = os.path.join("splits", f"train_oversampling{tag}.txt")
    write_list(out_path, out_list)
    final_ratio = out_tot[args.soft_id] / max(out_tot[args.hard_id], 1)
    print(f"\nWrote:\n  {out_path} (lines: {len(out_list)})")
    print(f"Approx totals after oversampling: {out_tot}")
    print(f"Final soft:hard ≈ {final_ratio:.3f}")
    print(f'Use with: --names "{args.tag or "oversampling"}:{out_path}"')


if __name__ == "__main__":
    main()


Now if we have everything ready, we can proceed with next steps.\
In this code below we are going to train model, we are using different traning data so user has to adjust it in line - 19 -  and change training path in data.yaml. User should train the model on each file beginning with "train" in the "splits" folder.


In [ ]:
def parse_args():
    """
    Parse command line arguments for YOLOv8 training.
    """
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="yolov8_sm_latest.pt", help="Path to YOLOv8 seg model .pt")
    ap.add_argument("--data", default="data.yaml", help="Path to base data.yaml (fixed val/test)")
    ap.add_argument("--runs", default="runs", help="Directory to store runs")
    ap.add_argument("--epochs", type=int, default=20)
    ap.add_argument("--imgsz", type=int, default=640)
    ap.add_argument("--batch", default="auto", help='int/float or "auto"')
    ap.add_argument("--freeze", type=int, default=10)
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--seeds", type=int, nargs="+", default=[0])
    ap.add_argument("--device", default=None, help="CUDA device index, e.g. 0; leave empty for auto")
    ap.add_argument(
        "--names",
        nargs="+",
        default=["baseline_realistic:splits/train_realistic.txt"],
        help="Pairs name:train_list.txt (space separated)"
    )
    return ap.parse_args()


def _normalize_batch(b):
    """
    Normalize batch parameter: convert 'auto' -> -1, or cast string to int/float.
    """
    if isinstance(b, (int, float)):
        return b
    if isinstance(b, str):
        if b.lower() == "auto":
            return -1
        try:
            return int(b)
        except ValueError:
            return float(b)
    return b


def main():
    """
    Main function to run YOLOv8 training with multiple seeds and dataset splits.
    """
    args = parse_args()

    if not os.path.exists(args.model):
        print(f"[ERROR] Model .pt not found: {args.model}")
        sys.exit(1)
    if not os.path.exists(args.data):
        print(f"[ERROR] data.yaml not found: {args.data}")
        sys.exit(1)
    os.makedirs(args.runs, exist_ok=True)

    pairs = {}
    for item in args.names:
        if ":" not in item:
            print(f"[ERROR] Bad --names item (expected name:path): {item}")
            sys.exit(1)
        name, train_list = item.split(":", 1)
        if not os.path.exists(train_list):
            print(f"[ERROR] Train list not found: {train_list}")
            sys.exit(1)
        pairs[name] = train_list

    with open(args.data, "r", encoding="utf-8") as f:
        base_yaml = f.read()

    batch_val = _normalize_batch(args.batch)

    for seed in args.seeds:
        for name, train_list in pairs.items():
            print(f"\n=== Starting training: {name} (seed {seed}) ===")
            tmp_yaml = os.path.join(os.path.dirname(args.data), f"data_{name}_s{seed}.yaml")
            lines, swapped = [], False
            for line in base_yaml.splitlines():
                if line.strip().lower().startswith("train:"):
                    lines.append(f"train: {train_list}")
                    swapped = True
                else:
                    lines.append(line)
            if not swapped:
                lines.append(f"train: {train_list}")
            with open(tmp_yaml, "w", encoding="utf-8") as f:
                f.write("\n".join(lines))

            run_name = f"{name}_s{seed}"
            try:
                model = YOLO(args.model)
                model.train(
                    data=tmp_yaml,
                    imgsz=args.imgsz,
                    epochs=args.epochs,
                    batch=batch_val,
                    freeze=args.freeze,
                    workers=args.workers,
                    seed=seed,
                    device=args.device,
                    project=args.runs,
                    name=run_name
                )
                print(f"✓ Completed: {run_name}")
            except Exception as e:
                print(f"[ERROR] Training failed for {run_name}: {e}")
                sys.exit(1)


if __name__ == "__main__":
    main()


After that user should have folder "runs" in which are saved all training data, which will be user to evaluate different approaches:

In [ ]:
import argparse, os, glob
import pandas as pd
from ultralytics import YOLO


def parse_args():
    """
    Parse command line arguments for evaluating YOLOv8 segmentation runs.
    """
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data.yaml", help="Path to data.yaml")
    ap.add_argument("--runs", default="runs", help="A run dir OR a parent dir containing multiple runs")
    ap.add_argument("--out", default="metrics_summary.csv", help="Output CSV")
    ap.add_argument("--split", default="test", choices=["val", "test"], help="Which split to evaluate on")
    ap.add_argument("--plots", action="store_true", help="Also save PR/F1 plots during val")
    return ap.parse_args()


def list_run_dirs(root: str):
    """
    Return a list of run directories containing weights/best.pt.
    """
    root = os.path.abspath(root)
    if os.path.isdir(os.path.join(root, "weights")) and os.path.exists(os.path.join(root, "weights", "best.pt")):
        return [root]
    kids = [p for p in glob.glob(os.path.join(root, "*")) if os.path.isdir(p)]
    runs = [p for p in kids if os.path.exists(os.path.join(p, "weights", "best.pt"))]
    if not runs:
        raise SystemExit(f"No run folders with weights/best.pt found under: {root}")
    return sorted(runs)


def try_get_seg_metrics(m):
    """
    Extract segmentation evaluation metrics from a YOLOv8 results object.
    """
    out = {}
    seg = getattr(m, "seg", None)
    if seg is not None:
        out["mAP50-95_all"] = float(getattr(seg, "map", float("nan")))
        out["mAP50_all"] = float(getattr(seg, "map50", float("nan")))
        out["per_class"] = list(getattr(seg, "maps", []))
        return out
    rd = getattr(m, "results_dict", None) or {}
    for k in ("metrics/seg_mAP50-95", "metrics/seg_map50-95", "metrics/mAP50-95(M)"):
        if k in rd:
            out["mAP50-95_all"] = float(rd[k])
            break
    for k in ("metrics/seg_mAP50", "metrics/seg_map50", "metrics/mAP50(M)"):
        if k in rd:
            out["mAP50_all"] = float(rd[k])
            break
    out["per_class"] = []
    return out


def main():
    """
    Evaluate YOLOv8 segmentation runs and save per-class + overall metrics to a CSV.
    """
    args = parse_args()
    runs = list_run_dirs(args.runs)

    any_best = os.path.join(runs[0], "weights", "best.pt")
    names = YOLO(any_best).names
    if isinstance(names, dict):
        class_names = [names[i] for i in range(len(names))]
    else:
        class_names = list(names)

    rows = []
    for rd in runs:
        best = os.path.join(rd, "weights", "best.pt")
        run_name = os.path.basename(rd)
        model = YOLO(best)
        metrics = model.val(data=args.data, split=args.split, plots=args.plots)
        seg = try_get_seg_metrics(metrics)

        row = {
            "run": run_name,
            "split": args.split,
            "mAP50-95_all": seg.get("mAP50-95_all", float("nan")),
            "mAP50_all": seg.get("mAP50_all", float("nan")),
        }

        per = seg.get("per_class", [])
        for i, cname in enumerate(class_names):
            row[f"mAP50-95_{cname}"] = float(per[i]) if i < len(per) else float("nan")

        if len(class_names) >= 2:
            a = row.get(f"mAP50-95_{class_names[0]}", float("nan"))
            b = row.get(f"mAP50-95_{class_names[1]}", float("nan"))
            try:
                row["MacroAP50-95"] = (a + b) / 2.0
            except Exception:
                row["MacroAP50-95"] = float("nan")

        rows.append(row)
        print(f"Evaluated {run_name}: mAP50-95_all={row['mAP50-95_all']:.3f}")

    df = pd.DataFrame(rows).sort_values(["run", "split"])
    out_path = os.path.abspath(args.out)
    out_dir = os.path.dirname(out_path) or "."
    os.makedirs(out_dir, exist_ok=True)
    df.to_csv(out_path, index=False)
    print(f"Wrote summary to {out_path}")


if __name__ == "__main__":
    main()
